# Line-of-sight projection and kernels

LOSProjector owns the redshift/comoving-distance quadrature and physical radial weights. It is independent of the downstream 3PCF route.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from fastnc.bispectrum import SPTMatterBispectrum3D
from fastnc.projection import Kernel1D, KernelSet, LOSProjector

b3d = SPTMatterBispectrum3D.simple_debug()

## Exact fixed-redshift projection

Use delta_like to debug one redshift slice. It evaluates the source exactly and does not approximate a delta function with a narrow window.

In [ ]:
fixed = LOSProjector.delta_like(z=0.5, chi=1300.0)
fixed_b2d = fixed.project(b3d)
print(fixed_b2d.evaluate(100.0, 120.0, 80.0))

## Finite-width source, lensing, and intrinsic-alignment kernels

The geometry below is intentionally compact. Scientific work should compute chi(z) consistently with the cosmology used by the bispectrum.

In [ ]:
z = np.linspace(0.05, 1.5, 96)
chi = 3000.0 * z / (1.0 + 0.45 * z)
z_source = np.linspace(0.05, 1.5, 160)
nz = z_source**2 * np.exp(-(z_source / 0.65) ** 1.5)

source = Kernel1D.from_nz(z, chi, z_source, nz, name="source")
lensing = Kernel1D.lensing_from_nz(
    z, chi, z_source, nz, omega_m=0.3, name="lensing"
)
intrinsic_alignment = Kernel1D.nla_from_nz(
    z, chi, z_source, nz, amplitude=1.0, omega_m=0.3,
    growth=lambda redshift: 1.0 / (1.0 + redshift), name="ia",
)
observed_shear = (lensing + intrinsic_alignment).copy(name="shear")

In [ ]:
for kernel in (source, lensing, intrinsic_alignment, observed_shear):
    plt.plot(kernel.z, kernel.weight, label=kernel.name)
plt.xlabel(r"$z$")
plt.ylabel("kernel weight")
plt.legend();

from_nz normalizes the input n(z) by default before converting it to n(chi). The resulting Kernel1D is not normalized a second time. Call normalized() only when that normalization belongs to the physical convention.

## Assemble named samples and project

In [ ]:
kernels = KernelSet({
    "source": source, "lensing": lensing,
    "ia": intrinsic_alignment, "shear": observed_shear,
})
projector = LOSProjector(z=z, chi=chi, kernels=kernels)

source_b2d = projector.project(
    b3d, sample_combination=("source", "source", "source"),
)
shear_b2d = projector.project(
    b3d, sample_combination=("shear", "shear", "shear"),
)
cross_b2d = projector.project(
    b3d, sample_combination=("lensing", "lensing", "source"),
)
weight = projector.weight(("shear", "shear", "shear"))
print(weight.shape, shear_b2d.evaluate(100.0, 120.0, 80.0))

Finite-width projection uses the geometrical prefactor $\chi^{-4}$ by default. Pass prefactor=1.0 only when the intended convention omits it. Projected bispectra remain directly evaluable and retain source/projector provenance, allowing structured routes to delay LOS integration when useful.